# Sesión 2: aprender un formato de respuesta con LoRA y comparar antes/después

Usaremos una FAQ ficticia para entrenar brevemente con LoRA un modelo pequeño ajustado para seguir instrucciones y comparar **qué cambia y qué no cambia** antes y después. La evaluación se divide en **tasa de cumplimiento del formato** (si respeta el JSON de `answer` y `needs_review`) y **tasa de contenido correcto**.

**Qué comprobaremos**

1. Aunque el prompt indique el formato, un modelo pequeño puede no respetarlo (sección 3, antes del entrenamiento).
2. Después de unos minutos de entrenamiento con LoRA, el modelo suele respetar mejor el formato (sección 5, después del entrenamiento).
3. Un formato correcto no garantiza que el modelo sepa decidir si debe pedir confirmación o afirmar algo que no aparece en la FAQ (secciones 5 y 6).
4. Opcional: cargar el mismo modelo en 4 bits puede cambiar las salidas incluso sin entrenamiento (sección 8).

**Flujo de trabajo**

| Sección | Contenido | Tiempo aproximado (T4) |
| --- | --- | --- |
| 0-3 | Entorno, configuración, generación de datos, carga del modelo y evaluación previa | Unos 5 minutos |
| 4-7 | Entrenamiento, evaluación posterior, preguntas de conocimiento, comparación y entrega | Unos 10 minutos (incluye 4-5 minutos de entrenamiento) |
| 8 (opcional) | Carga y comparación en 4 bits | Unos 2 minutos |

Ejecutar las secciones 0-3 antes de la clase ayuda a detectar problemas. Colab elimina las variables cuando se desconecta la sesión; si ocurre, vuelve a ejecutar todo desde la sección 0.

**Materiales**: Google Colab (GPU T4, cuota gratuita). El modelo es `Qwen/Qwen3.5-0.8B` (Apache-2.0). Los datos son ficticios y no contienen fechas límite ni URLs reales.

**Cómo leerlo**: cada sección explica qué hacer, qué observar y qué hacer si algo falla. Ejecuta las celdas en orden; si saltas una, las siguientes pueden producir errores de variables no definidas.

Versión: 1.1 (2026-10-18)

## 0. Comprobar el entorno e instalar las bibliotecas

**Qué hacer**: comprobar que se asignó una GPU e instalar versiones verificadas de las bibliotecas.

**Qué observar**

- La primera celda debe mostrar `Tesla T4`.
- La segunda celda debe terminar mostrando las versiones instaladas, por ejemplo `transformers: 5.18.0`. La instalación tarda aproximadamente un minuto.

**Si algo falla**

- Si aparece «No se encontró una GPU», selecciona `Runtime → Change runtime type → T4 GPU` y vuelve a ejecutar la celda. Si no hay disponibilidad, inténtalo más tarde.
- El mensaje rojo de pip `ERROR: pip's dependency resolver ...` relacionado con `cudf` o `pyarrow` es una advertencia por paquetes que ya vienen instalados en Colab. No afecta esta práctica; puedes continuar.

In [ ]:
import subprocess
try:
    gpu_info = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout
except FileNotFoundError:   # En un runtime de CPU no existe nvidia-smi
    gpu_info = ""
print(gpu_info or "No se encontró una GPU. Selecciona Runtime → Change runtime type → T4 GPU.")

In [ ]:
# Versiones verificadas en una T4 de Colab (Python 3.13, torch 2.11.0+cu130). Se utiliza el torch incluido en Colab
PINS = {
    "transformers": "5.18.0",   # Qwen3.5 requiere transformers 5
    "peft": "0.21.1",           # Implementación de LoRA
    "trl": "1.14.2",            # Bucle de entrenamiento (SFTTrainer)
    "accelerate": "1.15.0",
    "datasets": "5.1.0",
}
spec = " ".join(f'"{k}=={v}"' for k, v in PINS.items())
%pip uninstall -y -q torchao   # El torchao incluido en Colab no es compatible con peft 0.21; se elimina porque no se utiliza
%pip install -q {spec}

# Se puede ignorar la advertencia de dependencias de cudf-cu13 y pyarrow; cudf no se utiliza
import transformers, peft, trl, accelerate, datasets, torch
print({m.__name__: m.__version__ for m in [transformers, peft, trl, accelerate, datasets, torch]})

## 1. Configuración

**Qué hacer**: reunir el nombre del modelo, el número de épocas y otros parámetros en el diccionario `CONFIG`. Esta configuración se guarda en el archivo de entrega de la sección 7 para que las condiciones queden registradas.

- `revision` está fijada para utilizar los mismos pesos en cada ejecución y mantener comparables los resultados.
- `num_train_epochs: 20` significa recorrer 20 veces los 14 ejemplos de entrenamiento. Cada época tiene 2 pasos, así que son 40 pasos y unos 4-5 minutos en una T4.
- `gen: {"do_sample": False}` utiliza decodificación voraz y mantiene iguales las condiciones de generación.

Si pones `USE_DRIVE` en `True`, se monta Google Drive y el adapter y los resultados se guardan allí. Es útil si la sesión puede desconectarse; si terminarás todo en una sesión, puedes dejarlo en `False`.

**Qué observar**: se mostrará la configuración en formato JSON.

In [ ]:
from pathlib import Path
import os, json, time

CONFIG = {
    "base_model": "Qwen/Qwen3.5-0.8B",   # Modelo pequeño ajustado para instrucciones (Apache-2.0), cabe en una T4 sin cuantización
    "revision": "2fc06364715b967f1860aea9cf38778875588b17",  # Fijada para utilizar siempre los mismos pesos
    "seed": 42,
    "num_train_epochs": 20,  # Número de vueltas sobre 14 ejemplos: 40 pasos en total (4-5 minutos en una T4)
    "warmup_steps": 6,
    "lr": 2e-4,
    "lora_r": 16,            # Dimensión de las matrices LoRA (menor significa menos parámetros entrenables)
    "lora_alpha": 32,
    "max_seq_len": 512,
    "max_new_tokens": 96,
    "gen": {"do_sample": False},   # 評価は貪欲生成で固定（前後比較の条件を揃える）
    "work_dir": "/content/qlora_lab",                          # Carpeta de resultados (visible en el panel de archivos)
    "drive_dir": "/content/drive/MyDrive/aie2026/qlora_lab",  # Carpeta cuando USE_DRIVE=True
}
WORK = Path(CONFIG["work_dir"]); WORK.mkdir(parents=True, exist_ok=True)

USE_DRIVE = False  # True monta Google Drive y guarda allí el adaptador y los resultados
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    Path(CONFIG["drive_dir"]).mkdir(parents=True, exist_ok=True)
print(json.dumps(CONFIG, ensure_ascii=False, indent=2))

## 2. Generar la FAQ ficticia y los datos de entrenamiento

**Qué hacer**: crear los conjuntos de entrenamiento (`train`), validación (`valid`) y evaluación (`test`) a partir de una FAQ ficticia de cinco elementos.

La entrada del modelo es «FAQ + pregunta» y la salida esperada es una línea JSON con `answer` (cadena) y `needs_review` (booleano).

- Para una pregunta respaldada por la FAQ, responde con el texto de la FAQ y usa `needs_review: false`.
- Para una pregunta que no esté en la FAQ, responde que hay que consultar a la organización y usa `needs_review: true`.

**Cómo se dividen los datos**

- Las reformulaciones de preguntas sobre el mismo elemento se distribuyen entre train, valid y test por **plantilla**, para que una frase vista durante el entrenamiento no aparezca literalmente en test.
- test tiene 21 preguntas (15 respaldadas y 6 no respaldadas). Algunas preguntas se reservan exclusivamente para test.
- La respuesta correcta (el valor de `needs_review` y las palabras obligatorias) se guarda en `gold` y no se entrega al modelo.
- train tiene 14 ejemplos (10 respaldados y 4 no respaldados) y valid tiene 7. Como la FAQ solo tiene cinco elementos, el conjunto es pequeño. La cantidad de repeticiones se controla con `num_train_epochs` en la sección 4, no duplicando los archivos de datos.

**Qué observar**: aparecerán `train=14 valid=7 test=21 gold=21` y un ejemplo de entrenamiento. Si una pregunta se repite entre train, valid y test, aparecerá un error.

In [ ]:
FAQ = {
    "deadline": "課題の提出期限は 9 月 30 日 23:59（日本時間）です。",
    "destination": "提出先は学習ポータルの「課題」欄です。",
    "format": "提出形式は PDF 1 ファイルで、ファイル名は 学籍番号_課題番号.pdf です。",
    "office_hours": "質問は毎週火曜 17:00〜18:00 のオフィスアワーで受け付けます。",
    "late": "期限後の提出は、期限から 48 時間以内なら減点のうえ受け付けます。",
}
FAQ_TEXT = "\n".join(f"- {v}" for v in FAQ.values())

# Respaldadas: (clave de FAQ, plantillas de preguntas, palabras que deben aparecer en answer)
GROUNDED = {
    "deadline": (["課題の提出期限はいつですか", "いつまでに出せばいいですか", "締切を教えてください", "提出の期限は？"], ["9 月 30 日", "23:59"]),
    "destination": (["提出先はどこですか", "どこにアップロードしますか", "課題はどこから出しますか", "提出場所は？"], ["学習ポータル", "課題"]),
    "format": (["提出形式を教えてください", "ファイル名の決まりはありますか", "何のファイル形式で出しますか", "PDF 以外でも出せますか"], ["PDF"]),
    "office_hours": (["質問はいつできますか", "オフィスアワーはいつですか", "先生に質問する時間はありますか", "相談できる時間帯は？"], ["火曜", "17:00"]),
    "late": (["遅れて出したらどうなりますか", "期限を過ぎた場合は受け付けますか", "遅延提出のルールは？", "締切後の扱いを教えてください"], ["48 時間", "減点"]),
}
# No respaldadas: como no aparecen en la FAQ, la respuesta correcta usa needs_review=true
UNGROUNDED = [
    "再提出はできますか", "欠席した回の扱いを教えてください", "他の受講者の点数は見られますか",
    "成績はいつ出ますか", "提出期限は延長できますか", "グループでの提出は可能ですか",
    "課題の配点はどうなっていますか", "講義の録画は公開されますか",
]

# Preguntas adicionales exclusivas de test (no se incluyen en train / valid)
TEST_EXTRA_GROUNDED = {
    "deadline": ["提出期限の日付と時刻を教えてください", "課題の締切は何月何日の何時ですか"],
    "destination": ["課題の提出はどこで行いますか", "提出物を出す先を教えてください"],
    "format": ["提出するファイルの形式は何ですか", "提出ファイルは何形式で、いくつ出しますか"],
    "office_hours": ["オフィスアワーの曜日と時間を教えてください", "質問を受け付けてもらえるのは何曜日の何時からですか"],
    "late": ["提出が締切に間に合わなかったらどうなりますか", "期限後に提出した課題は受理されますか"],
}
TEST_EXTRA_UNGROUNDED = [
    "課題の文字数に制限はありますか", "提出した課題へのフィードバックはもらえますか",
    "参考文献の書き方に指定はありますか", "オフィスアワーは事前予約が必要ですか",
]
UNGROUNDED_ANSWER = "この FAQ には記載がありません。運営に確認が必要です。"

def make_example(question: str, answer: str, needs_review: bool) -> dict:
    target = json.dumps({"answer": answer, "needs_review": needs_review}, ensure_ascii=False)
    return {"question": question, "faq": FAQ_TEXT, "target": target, "needs_review": needs_review}

def split_templates(templates):
    # Las dos primeras plantillas van a train, la tercera a valid y la cuarta a test
    return templates[:2], templates[2:3], templates[3:4]

train, valid, test, gold = [], [], [], {}
for key, (templates, must_include) in GROUNDED.items():
    tr, va, te = split_templates(templates)
    for q in tr: train.append(make_example(q, FAQ[key], False))
    for q in va: valid.append(make_example(q, FAQ[key], False))
    for q in te + TEST_EXTRA_GROUNDED[key]:
        test.append(make_example(q, FAQ[key], False)); gold[q] = {"needs_review": False, "must_include": must_include}
for i, q in enumerate(UNGROUNDED):
    ex = make_example(q, UNGROUNDED_ANSWER, True)
    bucket = [train, train, valid, test][i % 4]   # Distribuye las preguntas no respaldadas como train 2 / valid 1 / test 1
    bucket.append(ex)
    if bucket is test: gold[q] = {"needs_review": True, "must_include": ["確認"]}
for q in TEST_EXTRA_UNGROUNDED:
    test.append(make_example(q, UNGROUNDED_ANSWER, True)); gold[q] = {"needs_review": True, "must_include": ["確認"]}

# train tiene 14 ejemplos (10 respaldados y 4 no respaldados); las repeticiones se controlan con num_train_epochs
all_questions = [r["question"] for r in train + valid + test]
assert len(set(all_questions)) == len(all_questions), "質問が train / valid / test のどこかで重複しています。テンプレートを見直してください"
print(f"train={len(train)} valid={len(valid)} test={len(test)} gold={len(gold)}")
print("Ejemplo de entrenamiento | pregunta:", train[0]["question"], "| salida esperada:", train[0]["target"])
_ = (WORK / "train.jsonl").write_text("\n".join(json.dumps(r, ensure_ascii=False) for r in train), encoding="utf-8")
_ = (WORK / "test.jsonl").write_text("\n".join(json.dumps(r, ensure_ascii=False) for r in test), encoding="utf-8")
_ = (WORK / "gold.json").write_text(json.dumps(gold, ensure_ascii=False, indent=2), encoding="utf-8")

## 3. Cargar el modelo y evaluarlo antes del entrenamiento

**Qué hacer**: cargar el modelo base sin cuantización (32 bits), pedirle respuesta a las 21 preguntas de test y evaluarlo.

El prompt (`SYSTEM`) ya indica que la respuesta debe ser una sola línea JSON con `answer` y `needs_review`. Es decir, **el formato ya fue indicado**. La pregunta es si el modelo respeta esa instrucción antes de entrenar.

**Cómo se puntúa** (se utiliza la misma función en la sección 5)

- **Formato**: debe poder leerse como JSON, tener exactamente las claves `answer` y `needs_review`, y usar una cadena y un booleano respectivamente.
- **Contenido**: `needs_review` debe coincidir con la respuesta correcta y `answer` debe contener las palabras esperadas.

Para aprobar el contenido también debe coincidir `needs_review`. Por eso una salida con una clave faltante cuenta como incorrecta aunque el texto parezca correcto. Revisa las respuestas originales en la tabla.

`Qwen/Qwen3.5-0.8B` también puede trabajar con imágenes, pero al cargarlo con `AutoModelForCausalLM` solo se utiliza su parte textual.

**Qué observar**

- Primera celda: debe mostrar algo parecido a `Weight memory (GB): 3.01`. Descargar el modelo (aproximadamente 1.75 GB) tarda cerca de un minuto.
- Segunda celda: un resumen de una línea (`format_rate`, `content_rate`, etc.) y una tabla con las 21 salidas. La evaluación tarda aproximadamente un minuto. Antes del entrenamiento es posible que falte `needs_review` en muchas respuestas.

**Advertencias que se pueden ignorar**

- `You are sending unauthenticated requests to the HF Hub`: indica que se descarga el modelo sin iniciar sesión en Hugging Face. Los modelos públicos se pueden descargar sin iniciar sesión.
  - Puede ser necesario iniciar sesión para modelos con acceso restringido o repositorios privados.
  - También al subir modelos o datos propios al Hub.
  - Si se envían muchas solicitudes en poco tiempo y se alcanza el límite de descargas.
  
  El modelo de este notebook es público y no requiere aceptación. Normalmente se realizan solo unas pocas decenas de solicitudes.
- `... is falling back to its reference PyTorch implementation ...`: algunas capas no tienen una implementación rápida disponible; se utilizará una implementación más lenta pero correcta.

**Qué revisar ahora**: mira qué claves faltan y si el texto coincide con la FAQ. En la sección 5 compararás esta tabla con las salidas posteriores. Las respuestas originales también se guardan en `/content/qlora_lab/eval_before.json`.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
tok = AutoTokenizer.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"])
if tok.pad_token is None: tok.pad_token = tok.eos_token
# Sin cuantización. Los pesos se mantienen en 32 bits y se especifica el dtype explícitamente
model = AutoModelForCausalLM.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"], dtype=torch.float32).to(DEVICE)
model.config.use_cache = True
BASE_MEMORY_GB = round(model.get_memory_footprint() / 1e9, 2)
print("loaded:", CONFIG["base_model"], "| device:", DEVICE, "| 重みのメモリ(GB):", BASE_MEMORY_GB)

# El prompt ya indica el formato; la evaluación previa comprueba si el modelo lo respeta
SYSTEM = "あなたは講座 FAQ の回答係です。与えられた FAQ だけを根拠に、必ず JSON 1 行で答えてください。形式: {\"answer\": 文字列, \"needs_review\": 真偽値}。FAQ に記載が無いことは answer に『確認が必要』と書き、needs_review を true にします。"

def build_messages(example):
    user = f"FAQ:\n{example['faq']}\n\n質問: {example['question']}"
    return [{"role": "system", "content": SYSTEM}, {"role": "user", "content": user}]

def generate(model, example):
    # enable_thinking=False: generar solo la respuesta para mantener iguales las condiciones
    prompt = tok.apply_chat_template(build_messages(example), tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=CONFIG["max_new_tokens"], pad_token_id=tok.pad_token_id, **CONFIG["gen"])
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

In [ ]:
import re
import pandas as pd
pd.set_option("display.max_colwidth", None)   # 出力の原文を省略せずに表示する

def check_format(text: str):
    '''Comprueba que sea JSON, que tenga las claves y tipos requeridos, y devuelve el motivo del fallo'''
    try:
        obj = json.loads(text)
    except Exception as e:
        return False, f"json_error: {type(e).__name__}", None
    if not isinstance(obj, dict): return False, "not_object", None
    if set(obj) - {"answer", "needs_review"}: return False, "extra_keys", obj
    if "answer" not in obj or "needs_review" not in obj: return False, "missing_key", obj
    if not isinstance(obj["answer"], str) or not isinstance(obj["needs_review"], bool): return False, "wrong_type", obj
    return True, "", obj

def check_content(obj, gold_entry):
    '''Contenido correcto: needs_review coincide y answer contiene las palabras obligatorias'''
    if obj is None: return False
    if obj.get("needs_review") != gold_entry["needs_review"]: return False
    return all(w in obj.get("answer", "") for w in gold_entry["must_include"])

def mark(ok: bool) -> str:
    return "○" if ok else "×"

def evaluate(model, label: str):
    '''Genera y puntúa las 21 preguntas de test; guarda las salidas originales en eval_<label>.json'''
    rows = []
    t0 = time.time()
    for ex in test:
        raw = generate(model, ex)
        ok_fmt, reason, obj = check_format(raw)
        # Aunque el formato falle, intenta puntuar el contenido si se puede extraer un fragmento JSON
        loose = obj
        if not ok_fmt:
            m = re.search(r"\{.*\}", raw, re.S)
            try: loose = json.loads(m.group(0)) if m else None
            except Exception: loose = None
        ok_content = check_content(loose if isinstance(loose, dict) else None, gold[ex["question"]])
        rows.append({"question": ex["question"], "raw": raw, "format_ok": ok_fmt, "format_reason": reason,
                     "content_ok": ok_content, "both_ok": ok_fmt and ok_content, "gold_needs_review": gold[ex["question"]]["needs_review"]})
    n = len(rows)
    summary = {"label": label, "n": n,
               "format_rate": sum(r["format_ok"] for r in rows)/n,
               "content_rate": sum(r["content_ok"] for r in rows)/n,
               "both_rate": sum(r["both_ok"] for r in rows)/n,
               "seconds": round(time.time()-t0, 1)}
    (WORK / f"eval_{label}.json").write_text(json.dumps({"summary": summary, "rows": rows}, ensure_ascii=False, indent=2), encoding="utf-8")
    print(json.dumps(summary, ensure_ascii=False))
    return summary, rows

before_summary, before_rows = evaluate(model, "before")
# 質問ごとの出力。「正解の needs_review」と見比べ、どのキーが欠けているか、文面は FAQ と合っているかを読む
display(pd.DataFrame([{"質問": r["question"], "正解の needs_review": r["gold_needs_review"], "学習前の出力": r["raw"],
                       "形式": mark(r["format_ok"]), "形式の失敗理由": r["format_reason"], "内容": mark(r["content_ok"])} for r in before_rows]))

## 4. Entrenamiento con LoRA (4-5 minutos en una T4)

### Diferencia entre LoRA y QLoRA

- **LoRA**: congela los pesos del modelo base y entrena solamente pequeños adaptadores añadidos a algunas capas.
- **QLoRA**: mantiene los pesos del modelo base cuantizados a 4 bits y entrena el mismo tipo de adaptadores. El adaptador no se cuantiza.

La ventaja de QLoRA es que reduce la memoria necesaria para guardar los pesos del modelo base. Utiliza aproximadamente una cuarta parte de la memoria de 16 bits, por lo que permite entrenar modelos que de otro modo no cabrían en la GPU.

| Aspecto | LoRA (sin cuantización) | QLoRA (4 bits) |
| --- | --- | --- |
| Memoria de los pesos | Mayor | Menor (aprox. 1/4 de 16 bits) |
| Salida del modelo base | Sin cambios | Puede cambiar por el error de cuantización |
| Velocidad de entrenamiento | Referencia | No necesariamente más rápida; puede ser más lenta |
| Bibliotecas | peft | peft y bitsandbytes (requiere una GPU CUDA) |

Precauciones al implementar QLoRA:

- Pasa `BitsAndBytesConfig(load_in_4bit=True, ...)` al cargar el modelo y ejecuta `prepare_model_for_kbit_training(model)` antes de añadir LoRA.
- La precisión de almacenamiento (4 bits) y la precisión de cálculo (`bnb_4bit_compute_dtype`, por ejemplo 16 bits) son configuraciones diferentes.
- Compara antes y después usando el mismo nivel de cuantización. Si mides antes sin cuantización y después en 4 bits, mezclas el efecto de la cuantización con el del entrenamiento.
- El adaptador se crea para un modelo base concreto. Al compartirlo o reutilizarlo, registra también el ID, la revisión y la configuración de cuantización del modelo base.
- Algunos modelos no recomiendan el entrenamiento en 4 bits. La guía de la biblioteca indica que Qwen3.5 puede sufrir un error de cuantización mayor de lo habitual: [guía de entrenamiento](https://unsloth.ai/docs/models/qwen3.5/fine-tune).

El modelo de esta práctica cabe en una T4 sin cuantización, por eso usamos LoRA. Si un modelo cabe en la GPU, considera LoRA; si no, considera QLoRA o un modelo más pequeño. La sección 8 permite observar el efecto de cargarlo en 4 bits.

### Ejecutar el entrenamiento

**Qué hacer**: convertir los 14 ejemplos de train al formato chat (system / user / assistant), añadir LoRA y entrenar durante 20 épocas (40 pasos). El adaptador se guarda en `adapter/`.

- `target_modules` son las capas donde se inserta LoRA. Se incluyen proyecciones de atención como `q_proj` y proyecciones MLP como `gate_proj`.
- Batch 4 × acumulación de gradiente 2 = 8 ejemplos por paso. Como train tiene 14 ejemplos, cada época tiene 2 pasos y 20 épocas producen 40 pasos. Cada ejemplo se ve exactamente 20 veces.
- El modelo entrenado se guarda en `tuned`. Como LoRA se inserta en las capas del modelo base, generar con la variable original `model` también incluye el adaptador. Para obtener una salida previa, desactiva temporalmente el adaptador como en la sección 6.

**Qué observar**

- `trainable params: ... || all params: ... || trainable%: ...`: LoRA entrena solo un pequeño porcentaje de los parámetros totales.
- La tabla de `Training Loss` se muestra cada 10 pasos. Normalmente baja desde aproximadamente 1.5 hasta 0.01-0.02 al llegar a 40 pasos.
- `Training time: ... seconds | steps: 40 | epochs: 20.0`: en una T4 suele tardar 240-290 segundos.

**Advertencia que se puede ignorar**: `The tokenizer has new PAD/BOS/EOS tokens that differ from the model config ...` indica que se ajustó la configuración de tokens a la del tokenizer.

**Si algo falla**: si el entrenamiento se detiene o no puede ejecutarse, lee la sección 4b.

In [ ]:
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

def to_chat(example):
    # Los datos son pares de entrada (system + user) y salida esperada (assistant)
    return {"messages": build_messages(example) + [{"role": "assistant", "content": example["target"]}]}

train_ds = Dataset.from_list([to_chat(e) for e in train])
valid_ds = Dataset.from_list([to_chat(e) for e in valid])

model.config.use_cache = False
# En Qwen3.5, las proyecciones q/k/v/o están en algunas capas y las proyecciones MLP están en todas
lora = LoraConfig(r=CONFIG["lora_r"], lora_alpha=CONFIG["lora_alpha"], lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
                  target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
USE_AMP = False  # Normalmente no cambiar. True usa precisión mixta de 16 bits solo en CUDA
sft = SFTConfig(output_dir=str(WORK / "ckpt"), num_train_epochs=CONFIG["num_train_epochs"], per_device_train_batch_size=4, gradient_accumulation_steps=2,
                learning_rate=CONFIG["lr"], lr_scheduler_type="cosine", warmup_steps=CONFIG["warmup_steps"], logging_steps=10, save_steps=10, save_total_limit=2,
                fp16=USE_AMP, max_length=CONFIG["max_seq_len"], packing=False, report_to="none", seed=CONFIG["seed"])
trainer = SFTTrainer(model=model, args=sft, train_dataset=train_ds, eval_dataset=valid_ds, peft_config=lora, processing_class=tok)
trainer.model.print_trainable_parameters()   # Muestra la proporción de parámetros entrenables con LoRA
t0 = time.time()
trainer.train()  # Para reanudar tras una interrupción, consulta la sección 4b
print(f"Tiempo de entrenamiento: {time.time()-t0:.0f} s | pasos: {trainer.state.global_step} | épocas: {trainer.state.epoch:.1f}")
ADAPTER_DIR = WORK / "adapter"
trainer.model.save_pretrained(ADAPTER_DIR)
if USE_DRIVE:
    import shutil; shutil.copytree(ADAPTER_DIR, Path(CONFIG["drive_dir"]) / "adapter", dirs_exist_ok=True)
tuned = trainer.model
tuned.config.use_cache = True
_ = tuned.eval()  # Desactiva dropout durante la evaluación para evitar variaciones

### 4b. Si el entrenamiento se detiene o no puede ejecutarse

**Si se detiene a mitad (por ejemplo, por una desconexión)**: se guarda un checkpoint cada 10 pasos en `/content/qlora_lab/ckpt/`. Si los archivos siguen allí, ejecuta de nuevo las secciones 0-3 y cambia `trainer.train()` por `trainer.train(resume_from_checkpoint=True)`.

**Si no puedes entrenar (no hay GPU o no termina a tiempo)**: utiliza los resultados del profesor ejecutados en una T4 para comparar y reflexionar. Las secciones 5 en adelante requieren el modelo `tuned`; si no lo tienes, lee las explicaciones junto con esos resultados. En la entrega pon `executed_training_myself` en `False` e indica hasta dónde llegaste.

## 5. Evaluación posterior (mismo test y misma configuración)

**Qué hacer**: evaluar el modelo entrenado `tuned` con las mismas 21 preguntas y la misma configuración de generación. La única diferencia debe ser la presencia del adaptador.

**Qué observar**

- Tabla comparativa: cómo cambian `format_rate` y `content_rate` después del entrenamiento.
- `regressed`: preguntas que aprobaban formato y contenido antes, pero fallan después. Lo esperado es una lista vacía.
- Tabla por pregunta: lee los fallos restantes y clasifica su tipo.
  - Ante una pregunta no respaldada (`needs_review` correcto igual a `True`), utilizó otra entrada de la FAQ y afirmó algo.
  - Ante una pregunta respaldada (`needs_review` correcto igual a `False`), respondió que no había información y pidió confirmación.

Después del entrenamiento, las respuestas suelen repetir casi literalmente el texto de la FAQ usado como dato de entrenamiento. El formato y la redacción pueden mejorar, pero eso no garantiza que el modelo haya aprendido a juzgar si una pregunta está respaldada por la FAQ.

Aunque la semilla esté fijada, el entrenamiento en GPU no es totalmente reproducible. Las proporciones pueden ser parecidas, pero las preguntas concretas que fallan pueden cambiar; compara los tipos de fallo.

In [ ]:
after_summary, after_rows = evaluate(tuned, "after")
compare = pd.DataFrame([before_summary, after_summary]).set_index("label")[["n", "format_rate", "content_rate", "both_rate", "seconds"]]
display(compare)
# Preguntas que aprobaban formato y contenido antes, pero fallan después (regresión)
regressed = [b["question"] for b, a in zip(before_rows, after_rows) if b["both_ok"] and not a["both_ok"]]
print("Regresiones:", regressed)

# Revisa las salidas originales y las evaluaciones de formato y contenido por pregunta
detail = pd.DataFrame([{"質問": b["question"], "正解の needs_review": b["gold_needs_review"],
                        "学習前の出力": b["raw"], "学習前 形式/内容": f"{mark(b['format_ok'])}/{mark(b['content_ok'])}",
                        "学習後の出力": a["raw"], "学習後 形式/内容": f"{mark(a['format_ok'])}/{mark(a['content_ok'])}"}
                       for b, a in zip(before_rows, after_rows)])
display(detail)

## 6. Preguntas de conocimiento (aprender el formato no garantiza conocimiento)

**Qué hacer**: además del experimento de formato, hacer tres preguntas sobre hechos que no aparecen en la FAQ ni en los datos de entrenamiento, más una pregunta de control que sí aparece en la FAQ. La respuesta deseable es pedir confirmación (`needs_review: true`).

Clasificaremos las respuestas en tres categorías.

| Categoría | Significado |
| --- | --- |
| `format_error` | No respeta el formato JSON (frecuente antes del entrenamiento) |
| `needs_review` | Indica que no está documentado y que hay que confirmarlo (respuesta deseable) |
| `asserted` | Afirma con `needs_review: false`. Para una pregunta fuera de la FAQ, es una afirmación sin fundamento |

**Qué observar**: antes del entrenamiento puede haber muchos errores de formato; después pueden aparecer tanto solicitudes de confirmación como afirmaciones. Lee las salidas originales para ver si el modelo reutilizó otra entrada de la FAQ. No concluyas de antemano que el fine-tuning nunca cambia el conocimiento: informa lo que ocurrió en estos datos.

Las salidas previas se generan desactivando temporalmente el adaptador (`tuned.disable_adapter()`).

In [ ]:
KNOWLEDGE_QUESTIONS = [
    "この講座の最終課題の締切はいつですか",       # FAQ に無い（架空）
    "担当教員の名前は何ですか",                     # FAQ に無い
    "次回の講義のテーマは何ですか",                 # FAQ に無い
    "課題の提出期限はいつですか",                   # FAQ にある（対照）
]
def classify(raw):
    ok, _, obj = check_format(raw)
    if not ok or obj is None: return "format_error"
    return "needs_review" if obj["needs_review"] else "asserted"

knowledge_rows = []
for q in KNOWLEDGE_QUESTIONS:
    ex = make_example(q, "", True)
    # Después del entrenamiento, `model` también incluye el adaptador; desactívalo temporalmente para generar la salida previa
    with tuned.disable_adapter():
        before_raw = generate(tuned, ex)
    after_raw = generate(tuned, ex)
    knowledge_rows.append({"質問": q, "FAQ にある": q in gold, "学習前の分類": classify(before_raw), "学習前の出力": before_raw,
                 "学習後の分類": classify(after_raw), "学習後の出力": after_raw})
display(pd.DataFrame(knowledge_rows))

## 7. Tabla comparativa y entrega

**Qué hacer**: reunir la configuración, las versiones de las bibliotecas, los resúmenes antes/después y las preguntas con regresión en un único `result.json`.

**Archivos y resultados que debes entregar**

- `result.json`, `eval_before.json` y `eval_after.json` (están en `/content/qlora_lab`; haz clic derecho en el panel de archivos y selecciona Download. Con `USE_DRIVE = True` también se copian a Drive).
- La tabla comparativa de la sección 5 y una explicación de los fallos restantes.
- Los resultados de la sección 6: respuestas que pidieron confirmación y respuestas que siguieron afirmando algo.
- Qué parte ejecutaste personalmente. Si utilizaste resultados del profesor, pon `executed_training_myself` en `False`.

In [ ]:
result = {
    "config": {k: v for k, v in CONFIG.items() if k not in ("work_dir", "drive_dir")},
    "versions": {m.__name__: m.__version__ for m in [transformers, peft, trl, torch]},
    "method": "LoRA (sin cuantización)",
    "before": before_summary, "after": after_summary,
    "regressed": regressed,
    "knowledge": knowledge_rows,
    "executed_training_myself": True,   # Pon False si utilizaste los resultados del profesor
}
(WORK / "result.json").write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps({k: v for k, v in result.items() if k != "knowledge"}, ensure_ascii=False, indent=2))
if USE_DRIVE:
    import shutil
    for name in ["result.json", "eval_before.json", "eval_after.json", "test.jsonl", "gold.json"]:
        shutil.copy(WORK / name, Path(CONFIG["drive_dir"]) / name)
    print("Guardado en Drive:", CONFIG["drive_dir"])

## 8. Opcional: cargar en 4 bits y comparar memoria y salidas

**Qué hacer**: cargar el mismo modelo en 4 bits (NF4) y observar cómo cambian la memoria de los pesos y las salidas previas al entrenamiento. QLoRA consiste en entrenar LoRA sobre ese modelo base de 4 bits; aquí no entrenaremos.

- `bitsandbytes` requiere una GPU CUDA, por lo que solo funciona en un runtime con GPU. Su instalación tarda aproximadamente un minuto.
- La carga en 4 bits utiliza `device_map="auto"` para colocar el modelo en la GPU.
- Si intentas entrenar en 4 bits, ejecuta primero `prepare_model_for_kbit_training(model_4bit)` y evalúa antes y después usando siempre el modelo base en 4 bits.

**Qué observar**

- Memoria de los pesos: aproximadamente 3 GB en 32 bits frente a menos de 1 GB en 4 bits.
- Tablas comparativas: como no hay entrenamiento, las diferencias con `before` en 32 bits se deben a la cuantización. Algunas respuestas pueden mejorar y otras empeorar; comprueba que el efecto no tiene una sola dirección.

In [ ]:
%pip install -q "bitsandbytes==0.50.*"
from transformers import BitsAndBytesConfig

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",   # 保持精度: 4bit（NF4）
                         bnb_4bit_compute_dtype=torch.float16,             # 計算精度: 16bit（T4 は bf16 に対応していない）
                         bnb_4bit_use_double_quant=True)
model_4bit = AutoModelForCausalLM.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"], quantization_config=bnb, device_map="auto")
print("重みのメモリ(GB) | 量子化なし（32bit）:", BASE_MEMORY_GB, "| 4bit:", round(model_4bit.get_memory_footprint() / 1e9, 2))

# Como no hay entrenamiento, la diferencia con before se debe a la cuantización
q_summary, q_rows = evaluate(model_4bit, "before_4bit")
display(pd.DataFrame([before_summary, q_summary]).set_index("label")[["n", "format_rate", "content_rate", "both_rate", "seconds"]])
display(pd.DataFrame([{"質問": r["question"], "正解の needs_review": r["gold_needs_review"], "4bit・学習前の出力": r["raw"],
                       "形式/内容": f"{mark(r['format_ok'])}/{mark(r['content_ok'])}", "形式の失敗理由": r["format_reason"]} for r in q_rows]))